# Notebook 3 – Poses and goals

In this notebook you work with **poses** (position + heading) and send the
robot to goals from Python.

You will:
1. convert between headings and quaternions,
2. ask TF and AMCL where the robot is (live),
3. check a goal against the map before sending it,
4. send goals to Nav2 with the `NavClient` from our package and plot the path
   the robot drove.

Tutorial: [chapter 3 – coordinate frames](../../TUTORIAL_SLAM_NAVIGATION.md#3-coordinate-frames-tf--the-heart-of-it-all),
[chapter 10 – driving to coordinates](../../TUTORIAL_SLAM_NAVIGATION.md#10-practice-2-driving-to-coordinates)

**How to use this notebook**

- Run the cells from top to bottom with **Shift + Enter**.
- `LIVE = False` uses the sample data in `sample_data/` – no robot needed.
- `LIVE = True` takes the data from the running robot. Start Jupyter from a
  terminal in which ROS 2 and the workspace are sourced (see `README.md` in this folder).
- Change values, run the cells again, look at what changes. You can't break anything.

**Live mode needs:** `robot.launch.py` on the Jetson and
`navigation.launch.py` on the PC (chapter 10, steps 1–5). Sections 1 and 2 also
work without a robot.

In [ ]:
%matplotlib inline
import math
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from nb_helpers import *

LIVE = False     # True: robot + navigation.launch.py are running
MAP_YAML = MY_MAP if LIVE else SAMPLE_MAP

## 1. Heading ↔ quaternion

ROS stores orientations as quaternions $(x, y, z, w)$. For a rotation about the
vertical axis only (our robot drives on the floor):

$$x = 0,\quad y = 0,\quad z = \sin(\psi / 2),\quad w = \cos(\psi / 2)$$

In [ ]:
print('  yaw [deg]      x      y       z       w')
for yaw_deg in [0, 45, 90, 180, -90]:
    qx, qy, qz, qw = yaw_to_quaternion(math.radians(yaw_deg))
    print(f'{yaw_deg:10.0f}  {qx:5.2f}  {qy:5.2f}  {qz:6.3f}  {qw:6.3f}')

`go_to` uses `make_pose()` from `nav_client.py` for exactly this. Let's check
that it gives the same result – and convert back:

In [ ]:
nav_client_class()                       # makes the imdl_nav package importable
from imdl_nav.nav_client import make_pose
from builtin_interfaces.msg import Time as TimeMsg

pose = make_pose(1.0, 0.5, 90, TimeMsg())
print(pose.pose)
print('back to degrees:', math.degrees(quaternion_to_yaw(pose.pose.orientation)))

**Question:** which quaternion belongs to 30°? Compute $z$ and $w$ by hand,
then check with `yaw_to_quaternion(math.radians(30))`.

## 2. Drawing poses

A pose is a point **and** a direction. Let's draw a few goal poses on the map.

In [ ]:
grid = GridMap(MAP_YAML)
config = load_robot_config()
radius = config['robot']['radius_cm'] / 100

poses = [(0.0, 0.0, 0), (1.0, 0.5, 90), (2.5, -0.5, 180), (4.0, 1.5, -90)]
fig, ax = plt.subplots(figsize=(10, 7))
show_map(ax, grid, 'goal poses: position + heading')
for x, y, yaw_deg in poses:
    draw_robot(ax, x, y, math.radians(yaw_deg), radius, color='tab:orange')
    ax.annotate(f'({x}, {y}, {yaw_deg} deg)', (x, y), xytext=(5, 12), textcoords='offset points')
plt.show()

## 3. Where is the robot? (live)

Two ways to ask:
- **TF**: look up `map → base_link`. This combines AMCL's correction
  `map → odom` with the odometry `odom → base_link`.
- **`/amcl_pose`**: AMCL's estimate, including its uncertainty (covariance).

In [ ]:
if LIVE:
    x, y, yaw = lookup_pose('map', 'base_link')
    print(f'map  -> base_link: x = {x:.3f} m, y = {y:.3f} m, heading = {yaw:.1f} deg')
    ox, oy, oyaw = lookup_pose('odom', 'base_link')
    print(f'odom -> base_link: x = {ox:.3f} m, y = {oy:.3f} m, heading = {oyaw:.1f} deg  (wheels only)')
    cx, cy, cyaw = lookup_pose('map', 'odom')
    print(f'map  -> odom     : x = {cx:.3f} m, y = {cy:.3f} m, heading = {cyaw:.1f} deg  (AMCL correction)')
else:
    print('LIVE = False: skipped')

In [ ]:
if LIVE:
    from geometry_msgs.msg import PoseWithCovarianceStamped
    from rclpy.qos import DurabilityPolicy, QoSProfile
    # AMCL publishes /amcl_pose as "transient local": we also get the last message
    latched = QoSProfile(depth=1, durability=DurabilityPolicy.TRANSIENT_LOCAL)
    msg = receive_one('/amcl_pose', PoseWithCovarianceStamped, latched)
    p = msg.pose.pose
    cov = np.array(msg.pose.covariance).reshape(6, 6)
    print(f'AMCL: x = {p.position.x:.3f} m, y = {p.position.y:.3f} m, '
          f'heading = {math.degrees(quaternion_to_yaw(p.orientation)):.1f} deg')
    print(f'uncertainty (1 sigma): x {math.sqrt(cov[0, 0]) * 100:.1f} cm, '
          f'y {math.sqrt(cov[1, 1]) * 100:.1f} cm, heading {math.degrees(math.sqrt(cov[5, 5])):.1f} deg')
else:
    print('LIVE = False: skipped')

Right after the start, AMCL may report an uncertainty of 0: it only updates its
estimate when the robot moves.

**Question:** drive the robot around a bit (with a goal below), then run both
cells of this section again. Which of the three transforms changes, and
why does `map → odom` not stay at zero?

## 4. Check a goal, then send it

Before sending a goal we check it against the map – like in notebook 2.
Then we use the `NavClient` from our package (`src/imdl_nav/imdl_nav/nav_client.py`)
– the same code as `ros2 run imdl_nav go_to`. We extend it a little so that it
remembers the robot pose from every feedback message.

**To cancel a drive, press the notebook's stop button (■).** The goal is then
canceled in Nav2 and the robot stops.

In [ ]:
NavClient = nav_client_class()


class RecordingNavClient(NavClient):
    """NavClient that also stores the robot pose from each feedback message."""

    def __init__(self):
        ros_node()                          # makes sure ROS is started
        super().__init__()
        self.track = []                     # (time, x, y, distance_remaining)
        self.last_print = 0.0

    def on_feedback(self, msg):
        f = msg.feedback
        p = f.current_pose.pose.position
        self.track.append((time.monotonic(), p.x, p.y, f.distance_remaining))
        if time.monotonic() - self.last_print > 1.0:
            print(f'  {f.distance_remaining:.2f} m to go')
            self.last_print = time.monotonic()


client = None                               # one client, reused for every drive


def drive(x, y, yaw_deg):
    """Check the goal, send it and return the recorded track."""
    global client
    distances = grid.distance_to_obstacle(1.0)
    ok, text = goal_check(grid, distances, x, y, config)
    print(text)
    if not ok:
        print('-> not sent.')
        return None
    if client is None:
        client = RecordingNavClient()
    client.track = []
    client.last_print = time.monotonic()
    print(f'driving to ({x}, {y}, {yaw_deg} deg) ...')
    try:
        success = client.drive_to(x, y, yaw_deg)
        print('reached!' if success else 'NOT reached (aborted or canceled)')
    except KeyboardInterrupt:
        client.cancel()
        print('canceled by you')
    return np.array(client.track)

In [ ]:
GOAL = (1.0, 0.5, 90)          # x [m], y [m], heading [deg] – change it!

if LIVE:
    track = drive(*GOAL)
else:
    print('LIVE = False: skipped (the check alone works offline):')
    print(goal_check(grid, grid.distance_to_obstacle(1.0), GOAL[0], GOAL[1], config)[1])
    track = None

In [ ]:
if track is not None and len(track):
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    show_map(axes[0], grid, 'path driven (from Nav2 feedback)')
    axes[0].plot(track[:, 1], track[:, 2], 'b-', linewidth=2, label='robot center')
    draw_robot(axes[0], GOAL[0], GOAL[1], math.radians(GOAL[2]), radius, color='tab:orange', label='goal')
    axes[0].legend(loc='upper right')
    axes[1].plot(track[:, 0] - track[0, 0], track[:, 3])
    axes[1].set_xlabel('time [s]'); axes[1].set_ylabel('distance remaining [m]'); axes[1].grid(True)
    plt.show()
    x, y, yaw = lookup_pose('map', 'base_link')
    print(f'final pose: ({x:.3f}, {y:.3f}, {yaw:.1f} deg) – goal was {GOAL}')
    print(f'position error: {math.hypot(x - GOAL[0], y - GOAL[1]) * 100:.1f} cm')

## 5. Several goals in a row

The same as `ros2 run imdl_nav waypoints`, but in Python, so you can add your
own logic between the goals.

In [ ]:
TOUR = [(1.0, 0.0, 0), (1.0, 0.5, 90), (0.0, 0.0, 0)]

if LIVE:
    for goal in TOUR:
        track = drive(*goal)
        if track is None:
            break
        time.sleep(2.0)            # e.g. "do something" at each goal
else:
    print('LIVE = False: skipped')

## Exercises

1. Send a goal 15 cm in front of a wall. What does `goal_check` say, and what
   does Nav2 do if you send it anyway (`client.drive_to(...)` directly)?
2. Measure the accuracy: drive to the same goal 3 times from different start
   points. How large is the position error each time? Compare it with
   `xy_goal_tolerance` in `config/nav2.yaml`.
3. Extend the tour: at each goal, print the AMCL uncertainty (section 3).
   Where is the robot most unsure about its position?
4. Write a function `square(side_m)` that drives a square with the given side
   length starting at the current robot pose (use `lookup_pose`).